In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/q10_rewrite/checkpoints/pre_cell_4.pickle

In [4]:
%%RecordEvent
%%time
### cell 4 ###

import datetime

# Define date bounds
date1 = datetime.datetime(1994, 11, 1)
date2 = datetime.datetime(1995, 2, 1)

# One‐chain of vectorized GPU operations: filter, project, join, compute revenue, aggregate & sort
total = (
    lineitem
      .loc[lineitem.L_RETURNFLAG == 'R', ['L_ORDERKEY', 'L_EXTENDEDPRICE', 'L_DISCOUNT']]
      .merge(
          orders.loc[(orders.O_ORDERDATE >= date1) & (orders.O_ORDERDATE < date2), ['O_ORDERKEY', 'O_CUSTKEY']],
          left_on='L_ORDERKEY',
          right_on='O_ORDERKEY'
      )
      .merge(
          customer[['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'C_NATIONKEY', 'C_ADDRESS', 'C_COMMENT']],
          left_on='O_CUSTKEY',
          right_on='C_CUSTKEY'
      )
      .merge(
          nation[['N_NATIONKEY', 'N_NAME']],
          left_on='C_NATIONKEY',
          right_on='N_NATIONKEY'
      )
      .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
      .groupby(
          ['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'N_NAME', 'C_ADDRESS', 'C_COMMENT'],
          as_index=False,
          sort=False
      )['TMP']
      .sum()
      .sort_values('TMP', ascending=False)
)


CPU times: user 87.7 ms, sys: 43.9 ms, total: 132 ms
Wall time: 131 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q10/rewritten/o4_mini_high_small/checkpoints/post_cell_4_try_2.pickle